# Experiment 5 -- aggregate and plot: success and failure modes of the pooled method

Reads every per-task CSV written by `run_experiment5.py` into
`Results_simulation/experiment5/raw/` (one file per `(target_regime,
regime, seed)`, each with `3 methods x len(MU_GRID)` rows: target,
source, pooled -- no adaptive, since this experiment isolates the pooled
method's own behavior rather than comparing it to the validation-
statistic switch), aggregates the Monte Carlo mean and standard error at
each `(target_regime, regime, mu, method)` grid point, and reproduces the
paper's supplementary Figure 7 (Appendix A.3, "Validation-free clustering
through a pooled estimator") as a 2 (target_regime) x 3 (regime) grid of
error-vs-alignment panels.

**Why this experiment exists:** the target+source pooled-subspace
estimator (`target_source_pooled_subspace_estimate`, rank-capped and
UNNORMALIZED -- `restrict_basis_rank=True, normalize=False` -- matching
the paper's Algorithm 4) can collapse to the target-only estimate whenever
the target's own (in-sample) direction estimate has larger raw magnitude
than the source's -- a structural winner-take-all effect of the rank-1
SVD used to combine the direction estimates, independent of how
well-aligned the source actually is. This experiment studies that
behavior directly, on purpose, by crossing Experiment 1's three
aspect-ratio regimes (R1, R2, R3) against two target-SNR scenarios:

- **`below_bbp`**: `Delta_T = 0.8` (Experiment 1's value) -- below the
  target-only recovery threshold `max(1, (d/n_T)^(1/4))` in every regime
  here (1.778 for R1/R2's `n_T=200, d=2000`; 1.0 for R3's `n_T=2500,
  d=2000`, since `d/n_T<1` there).
- **`above_bbp`**: `Delta_T = 2.0` -- comfortably above that threshold in
  every regime, while still below `Delta_S=3.0` (same as Experiment 1),
  i.e. the target is recoverable on its own but its SNR is still weaker
  than the source's.

Both `Delta_T` values are threaded through the *same* `(u,v)` draw per
`(regime, seed)` (see `run_experiment5.py`'s docstring), so the two
`target_regime` panels for a given regime are a paired comparison, not
independent draws.

Run this after the SLURM array in
`Slurm_Scripts/experiment5_pooled_modes/` has finished (or partially
finished -- it works fine on a subset of tasks).

In [ ]:
import sys, os, glob

# Hardcoded (rather than relative to "..") because the kernel's cwd isn't
# guaranteed to be this notebook's directory -- e.g. VS Code's Jupyter
# extension often starts kernels from the workspace root instead.
# EDIT: set this to the local path of your clone of this repository.
PROJECT_ROOT = "/path/to/Transfer_clustering"
sys.path.insert(0, os.path.join(PROJECT_ROOT, "Numerical_Experiments", "Experiments_Script"))

# The figure below uses matplotlib's text.usetex=True, which shells out to
# `latex`/`dvipng`. If a TeX Live install isn't already on PATH, set
# TEXLIVE_BIN to its bin/ directory (e.g. the output of `dirname $(which latex)`).
TEXLIVE_BIN = None
if TEXLIVE_BIN and os.path.isdir(TEXLIVE_BIN) and TEXLIVE_BIN not in os.environ["PATH"].split(os.pathsep):
    os.environ["PATH"] = TEXLIVE_BIN + os.pathsep + os.environ["PATH"]

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from run_experiment5 import (
    REGIME_ORDER, REGIMES, MU_GRID, TARGET_REGIME_ORDER, TARGET_REGIMES, bbp_threshold,
)

In [ ]:
RAW_DIR = os.path.join(PROJECT_ROOT, "Results_simulation", "experiment5", "raw")
COMBINED_DIR = os.path.join(PROJECT_ROOT, "Results_simulation", "experiment5", "combined")
os.makedirs(COMBINED_DIR, exist_ok=True)

paths = sorted(glob.glob(os.path.join(RAW_DIR, "*.csv")))
print(f"Found {len(paths)} raw result files")
assert paths, f"No CSVs found in {RAW_DIR} -- has the SLURM array finished any tasks yet?"

df = pd.concat([pd.read_csv(p) for p in paths], ignore_index=True)
df.head()

In [ ]:
# Sanity check: how many distinct seeds actually landed per (target_regime,
# regime, method)? If this is well short of N_SEEDS in run_experiment5.py,
# the array job hasn't finished (or some tasks failed -- check
# Slurm_Scripts/experiment5_pooled_modes/Error_Messages).
df.groupby(["target_regime", "regime", "method"])["seed"].nunique().unstack()

In [ ]:
summary = (
    df.groupby(["target_regime", "regime", "mu", "method"])["error"]
      .agg(mean_error="mean", se_error=lambda s: s.std(ddof=1) / np.sqrt(len(s)), n="count")
      .reset_index()
)
combined_path = os.path.join(COMBINED_DIR, "experiment5_summary.csv")
summary.to_csv(combined_path, index=False)
print(f"Wrote {combined_path}")
summary.head(12)

## BBP recovery threshold per regime

`max(1, (d/n_T)^(1/4))` for each aspect-ratio regime, alongside the
`Delta_T` actually used in each `target_regime` scenario (recorded
directly in the raw CSVs' `bbp_threshold`/`Delta_T` columns, but recomputed
here from `REGIMES`/`TARGET_REGIMES` for a compact reference table).
`below_bbp` should sit below the threshold in every row; `above_bbp`
should sit above it (and below `Delta_S=3.0`) in every row.

In [ ]:
threshold_table = pd.DataFrame(
    [
        dict(
            regime=regime,
            d=REGIMES[regime]["d"],
            n_T=REGIMES[regime]["n_T"],
            bbp_threshold=bbp_threshold(REGIMES[regime]["d"], REGIMES[regime]["n_T"]),
            **{f"Delta_T_{tr}": TARGET_REGIMES[tr]["Delta_T"] for tr in TARGET_REGIME_ORDER},
        )
        for regime in REGIME_ORDER
    ]
)
threshold_table

## Figure: error versus alignment, below- vs above-BBP target SNR

Rows = `target_regime` (`below_bbp` on top, `above_bbp` on bottom),
columns = aspect-ratio regime (R1, R2, R3). Each panel: `mu` on the
x-axis, Monte Carlo mean target misclustering error on a shared y-axis
(so magnitudes are directly comparable across every panel), one line per
method with SE error bars. Colors are fixed per method across all six
panels (never reassigned): blue = target-only, orange = source-only,
purple = pooled (rank-capped, unnormalized). Panel titles show the actual
`Delta_T` and BBP threshold for that (target_regime, regime) cell.

In [ ]:
METHOD_COLORS = {"target": "#2a78d6", "source": "#eb6834", "pooled": "#8e44ad"}
METHOD_LABELS = {"target": "Target-only", "source": "Source-only", "pooled": "Pooled"}
METHOD_ORDER = ["target", "source", "pooled"]

TARGET_REGIME_TITLES = {"below_bbp": "below BBP", "above_bbp": "above BBP"}

# Smallest nonzero mu on the grid sets the linear-to-log crossover, so
# mu=0 still shows up (symlog can't take log of 0) while everything else
# is spaced by log2(mu).
LINTHRESH = min(mu for mu in MU_GRID if mu > 0)

plt.rcParams.update({
    "text.usetex": True,
    "font.family": "serif",
    "font.size": 10,
    "axes.labelsize": 10,
    "axes.titlesize": 9,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
})

fig, axes = plt.subplots(len(TARGET_REGIME_ORDER), len(REGIME_ORDER), figsize=(13, 8), sharey=True)

for row, target_regime in enumerate(TARGET_REGIME_ORDER):
    Delta_T = TARGET_REGIMES[target_regime]["Delta_T"]
    for col, regime in enumerate(REGIME_ORDER):
        ax = axes[row, col]
        cfg = REGIMES[regime]
        threshold = bbp_threshold(cfg["d"], cfg["n_T"])
        sub = summary[(summary["target_regime"] == target_regime) & (summary["regime"] == regime)]
        for method in METHOD_ORDER:
            m = sub[sub["method"] == method].sort_values("mu")
            ax.errorbar(
                m["mu"], m["mean_error"], yerr=m["se_error"],
                label=METHOD_LABELS[method], color=METHOD_COLORS[method],
                linewidth=2, marker="o", markersize=5, capsize=3,
            )
        ax.set_title(
            rf"\texttt{{{regime}}} ({TARGET_REGIME_TITLES[target_regime]}): "
            rf"$\Delta_T={Delta_T}$, thr$={threshold:.3f}$"
        )
        ax.set_xscale("symlog", base=2, linthresh=LINTHRESH)
        if row == len(TARGET_REGIME_ORDER) - 1:
            ax.set_xlabel(r"$\mu$ (log$_2$ scale)")
        ax.spines[["top", "right"]].set_visible(False)
        ax.grid(True, alpha=0.25)

axes[0, 0].set_ylabel("target misclustering error")
axes[1, 0].set_ylabel("target misclustering error")
axes[0, -1].legend(frameon=False, fontsize=9, loc="upper right")
fig.suptitle("Experiment 5: Comparison of the pooled method with target and source only clustering", y=1.01)
fig.tight_layout()
fig.savefig(os.path.join(COMBINED_DIR, "experiment5_error_vs_alignment.pdf"), bbox_inches="tight")
plt.show()

## Figure: below-BBP panel

Same data and layout as the top row of the combined grid above
(`target_regime == "below_bbp"`), rendered as its own standalone figure
(a 1x3 row of R1/R2/R3 subplots) and saved to its own PDF. The
`above_bbp` panel is not produced here -- only `below_bbp` is presented.
Colors/labels/log-scale match the combined grid exactly; panel titles
carry only the regime, `Delta_T`, and BBP threshold (no "below BBP" /
"above BBP" tag, since there's nothing left to distinguish it from).

In [ ]:
target_regime = "below_bbp"
Delta_T_sep = TARGET_REGIMES[target_regime]["Delta_T"]

fig, axes = plt.subplots(1, len(REGIME_ORDER), figsize=(13, 4), sharey=True)
for ax, regime in zip(axes, REGIME_ORDER):
    cfg = REGIMES[regime]
    threshold = bbp_threshold(cfg["d"], cfg["n_T"])
    sub = summary[(summary["target_regime"] == target_regime) & (summary["regime"] == regime)]
    for method in METHOD_ORDER:
        m = sub[sub["method"] == method].sort_values("mu")
        ax.errorbar(
            m["mu"], m["mean_error"], yerr=m["se_error"],
            label=METHOD_LABELS[method], color=METHOD_COLORS[method],
            linewidth=2, marker="o", markersize=5, capsize=3,
        )
    ax.set_title(rf"\texttt{{{regime}}}: $\Delta_T={Delta_T_sep}$, thr$={threshold:.3f}$")
    ax.set_xscale("symlog", base=2, linthresh=LINTHRESH)
    ax.set_xlabel(r"$\mu$ (log$_2$ scale)")
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(True, alpha=0.25)

axes[0].set_ylabel("target misclustering error")
axes[-1].legend(frameon=False, fontsize=9, loc="upper right")
fig.suptitle("Experiment 5: Comparison of the pooled method with target and source only clustering", y=1.03)
fig.tight_layout()

out_path = os.path.join(COMBINED_DIR, f"experiment5_error_vs_alignment_{target_regime}.pdf")
fig.savefig(out_path, bbox_inches="tight")
plt.show()
print(f"Wrote {out_path}")